In [1]:
import torch
import triton
import triton.language as tl


@triton.jit

def SwiGLU_kernel(
    x_ptr,
    g_ptr,
    y_ptr,
    n_elements,
    BLOCK_SIZE: tl.constexpr
):
  pid=tl.program_id(0)

  one_block=pid*BLOCK_SIZE

  offsets=one_block+tl.arange(0, BLOCK_SIZE)

  mask=offsets<n_elements

  x=tl.load(x_ptr+offsets, mask=mask)
  g=tl.load(g_ptr+offsets, mask=mask)

  silu_g=g*tl.sigmoid(g)
  swiGLU=x*silu_g

  tl.store(y_ptr+offsets, swiGLU, mask=mask)

In [3]:
def SwiGLU_fused(x, g):
  y=torch.empty_like(x)
  n_elements=x.numel()

  grid=lambda meta: (triton.cdiv(n_elements, meta["BLOCK_SIZE"]),)

  SwiGLU_kernel[grid](
      x,
      g,
      y,
      n_elements,
      BLOCK_SIZE=1024,
  )
  return y